# Download protocol (assay) data

Test-drive [`get_protocol_data.py`](../python/get_protocol_data.py): pull each
protocol's readout data + SMILES into one **wide** table (one row per compound,
**latest** run per compound), selecting assays by their config `alias`.

> **Privacy:** the returned DataFrame holds SMILES and readout values. Any cell
> that renders rows (`df.head()`, echoing `df`) writes them into notebook output.
> **Clear all outputs before committing** (`jupyter nbconvert --clear-output
> --inplace vignettes/sample_protocol_download.ipynb`). Prefer `df.shape` /
> `df.columns` for quick checks.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.insert(0, os.path.abspath('../python'))

import pandas as pd
from get_protocol_data import get_data, load_config, alias_map

## paths are relative to this notebook (vignettes/)
CONFIG = '../config/config.yaml'
TOKEN_FILE = '~/.cdd_token'   # a READ token is enough to extract

## 1. Available experiments
The `alias -> (pid, name)` map comes straight from `config.yaml`.

In [2]:
alias_map(load_config(CONFIG))

{'mdck': (131399, 'MDR1-MDCK II Inhibitor'),
 'logd': (85979, 'LogD'),
 'solubility': (86015, 'Thermodynamic Solubility'),
 'ppb': (125963, 'Plasma protein binding (UC)'),
 'hlm': (88861, 'Microsomal stability species human'),
 'mlm': (88862, 'Microsomal stability species mouse'),
 'caco2': (85977, 'Caco-2 permeability'),
 'rlm': (111161, 'Microsomal stability species rat')}

## 2. Download one experiment
Returns a DataFrame: `molecule`, `name`, `smiles`, then `logd_run_date` +
`logd_<readout>` columns.

In [7]:
df_logd = get_data(experiments=['mdck'], config_path=CONFIG, token_file=TOKEN_FILE)
print('shape:', df_logd.shape)
df_logd.head(2)

experiments=['mdck']
  mdck (pid=131399): rows=114 compounds=112 readouts=26
compounds=112 columns=30
shape: (112, 30)


,molecule,name,smiles,mdck_run_date,mdck_Study number,mdck_Date,mdck_Mean Papp A to B,mdck_Mean Papp B to A,mdck_Efflux ratio,mdck_Mean %Solution Recovery A to B,...,mdck_Mean Papp A to B - PgP Inhibitor,mdck_Mean Papp B to A - PgP inhibitor,mdck_Efflux ratio - PgP inhibitor,mdck_Mean %Solution Recovery A to B - PgP inhibitor,mdck_Mean %Solution Recovery B to A - PgP inhibitor,mdck_Provider Comment - PgP inhibitor,mdck_Serac Comment - PgP inhibitor,mdck_Permeability Class - PgP inhibitor,mdck_Pgp substrate - Pgp inhibitor,mdck_Provider Name - PgP inhibitor
0,148093847,SRB-0002248,O=C1[C@@H]2CC3=C(C=CC(C#N)=C3)[C@@H](N2C(OCC2=...,2026-06-04,426354-20251024-MDR1,NaN,1.57,10.6,6.79,85.7,...,2.85,3.7,1.3,98.2,101.0,---,NaN,---,None,wuxi
1,151218893,SRB-0003109,FC1=C2[C@@H]3CNC(=O)[C@@H](N3C(C)=O)CC2=CC(F)=...,2026-06-04,426354-2025110602-MDR1,NaN,0.30,15.0,50.70,84.9,...,1.46,3.2,2.2,92.4,99.6,---,NaN,---,None,wuxi


## 3. Several experiments -> one merged wide table
One row per compound, joined on molecule id; each assay's readouts are
alias-prefixed so they never collide.

In [5]:
df = get_data(experiments=['logd', 'solubility', 'ppb'],
              config_path=CONFIG, token_file=TOKEN_FILE)
print('shape:', df.shape)
df.head(3)

experiments=['logd', 'solubility', 'ppb']
  logd (pid=85979): rows=324 compounds=319 readouts=6
  solubility (pid=86015): rows=124 compounds=121 readouts=8
  ppb (pid=125963): rows=91 compounds=80 readouts=9
compounds=320 columns=29
shape: (320, 29)


,molecule,name,smiles,logd_run_date,logd_Study number,logd_Study date,logd_LogD7.4,logd_Provider comment,logd_Provider name,logd_Serac comment,...,ppb_run_date,ppb_Study number,ppb_Date,ppb_Species,ppb_%Unbound,ppb_%Bound,ppb_Provider Comment,ppb_Provider Name,ppb_Serac Comment,ppb_%Remaining
0,121262866,SRB-0000069,C1=CC(NC2=C(C(=O)NOC[C@@H](CO)O)C=CC(F)=C2F)=C...,2023-06-09,426354-20230503-LogDY,20230509,3.00,None,WuXi,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,121308429,SRB-0000096,NC([C@@H](NC([C@@H](NC(CCOCCCONC(C1=CC=C(F)C(F...,2023-06-09,426354-20230503-LogDY,20230509,4.01,None,WuXi,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,121308431,SRB-0000098,CCC(N1C(C(N)=O)CC2=C(C=CC=C2)C1)=O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Download everything
Omit `experiments` to pull every aliased protocol (mdck, logd, hlm, mlm, rlm,
solubility, ppb, caco2).

In [ ]:
df_all = get_data(config_path=CONFIG, token_file=TOKEN_FILE)
print('shape:', df_all.shape)
list(df_all.columns)

### Add molecular descriptors (optional)
`mol_columns` accepts any molecule top-level field.

In [ ]:
df_desc = get_data(experiments=['logd', 'solubility'],
                   mol_columns=['name', 'smiles', 'inchi_key', 'molecular_weight', 'log_p'],
                   config_path=CONFIG, token_file=TOKEN_FILE)
print('shape:', df_desc.shape)
list(df_desc.columns)

## 5. Save to CSV
`output/` is local-only (gitignored).

In [ ]:
os.makedirs('../output', exist_ok=True)
df.to_csv('../output/protocol_data.csv', index=False)
print('wrote ../output/protocol_data.csv', df.shape)

## 6. Checks
Run after the download cells. Tweak without re-downloading.

In [ ]:
## one row per compound
assert df['molecule'].is_unique, 'expected one row per molecule'
## molecule identity columns present
assert {'molecule', 'name', 'smiles'} <= set(df.columns)
## each requested assay contributed alias-prefixed columns
for a in ['logd', 'solubility', 'ppb']:
    assert any(c.startswith(a + '_') for c in df.columns), f'no columns for {a}'
print('checks passed:', df.shape)